# 1. Setup and Libraries

In [1]:
import os
import glob
import json
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import concurrent.futures
from scipy.spatial.transform import Rotation
from pathlib import Path
from tqdm import tqdm
import time
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# 2. Dataset Directories

In [2]:
dataset_Folder = "G:/My Drive/GP Datasets/fit3d_data/"
file_path = dataset_Folder + 'fit3d_info.json'

# 3. Load FIT3D Metadata

In [3]:
info_JSON = pd.read_json(file_path, typ='series')
print("JSON loaded successfully as a series:")
print(info_JSON.head())

val_subj_names = ['s03', 's11']
all_train = [sub for sub in info_JSON['train_subj_names'] if sub not in val_subj_names]
all_camera_names = info_JSON['all_camera_names']

# Extract all actions from all subjects and find unique values
all_actions_list = [action for actions in info_JSON['subj_to_act'].values() for action in actions]
unique_actions = sorted(list(set(all_actions_list)))

print(f"Number of unique actions: {len(unique_actions)}")
print("Unique actions:")
print(unique_actions)

JSON loaded successfully as a series:
subj_to_act         {'s03': ['band_pull_apart', 'dumbbell_high_pul...
test_subj_names                                       [s02, s12, s13]
train_subj_names             [s03, s04, s05, s07, s08, s09, s10, s11]
all_camera_names             [50591643, 58860488, 60457274, 65906101]
dtype: object
Number of unique actions: 47
Unique actions:
['band_pull_apart', 'barbell_dead_row', 'barbell_row', 'barbell_shrug', 'burpees', 'clean_and_press', 'deadlift', 'diamond_pushup', 'drag_curl', 'dumbbell_biceps_curls', 'dumbbell_curl_trifecta', 'dumbbell_hammer_curls', 'dumbbell_high_pulls', 'dumbbell_overhead_shoulder_press', 'dumbbell_reverse_lunge', 'dumbbell_scaptions', 'man_maker', 'mule_kick', 'neutral_overhead_shoulder_press', 'one_arm_row', 'overhead_extension_thruster', 'overhead_trap_raises', 'pushup', 'side_lateral_raise', 'squat', 'standing_ab_twists', 'w_raise', 'walk_the_box', 'warmup_1', 'warmup_10', 'warmup_11', 'warmup_12', 'warmup_13', 'warmup_14

# 5. Mediapipe 3D Pose Evaluation
## 5.1 Configuration & Indices

In [4]:
# MediaPipe Pose setup
mp_pose = mp.solutions.pose

# Joint subset ordered by H36M joint index.
# Joint:        [ L_Hip, L_Knee, L_Ankle,  R_Hip, R_Knee, R_Ankle,  L_Shoulder, L_Elbow, L_Wrist, R_Shoulder, R_Elbow, R_Wrist]
fit3d_indices = [    1,      2,      3,     4,      5,       6,            11,     12,     13,          14,      15,       16]
mp_indices    = [   23,     25,      27,    24,     26,      28,           11,      13,      15,         12,      14,      16]

## 5.2 Utilities (Procrustes Alignment & Metric)

In [5]:
def procrustes_alignment_batched(pred, target):
    """
    Batched Procrustes alignment.
    pred, target: (num_frames, num_joints, 3)
    """
    mu_p = np.mean(pred, axis=1, keepdims=True)
    mu_t = np.mean(target, axis=1, keepdims=True)
    
    p_0 = pred - mu_p
    t_0 = target - mu_t
    
    norm_p = np.linalg.norm(p_0, axis=(1, 2), keepdims=True)
    norm_t = np.linalg.norm(t_0, axis=(1, 2), keepdims=True)
    norm_t = np.where(norm_t == 0, 1e-8, norm_t)
    norm_p = np.where(norm_p == 0, 1e-8, norm_p)
    
    p_0 = p_0 / norm_p
    t_0 = t_0 / norm_t
    
    # Kabsch algorithm (batched)
    H = np.matmul(p_0.transpose(0, 2, 1), t_0)
    U, S, Vt = np.linalg.svd(H)
    
    detR = np.linalg.det(np.matmul(Vt.transpose(0, 2, 1), U.transpose(0, 2, 1)))
    V_adj = Vt.transpose(0, 2, 1).copy()
    V_adj[detR < 0, :, 2] *= -1
    R = np.matmul(V_adj, U.transpose(0, 2, 1))
    
    aligned = np.matmul(p_0, R.transpose(0, 2, 1)) * norm_t + mu_t
    return aligned

def calculate_pa_mpjpe(pred_3d, gt_3d, mp_idx, gt_idx):
    """
    PA-MPJPE (Procrustes-Aligned MPJPE) — the primary evaluation metric.
    Removes the effect of global rotation, translation, and scale.
    pred_3d, gt_3d: full joint arrays in mm
    """
    num_frames = min(len(pred_3d), len(gt_3d))
    if num_frames == 0: return 0.0
    
    p_subset = np.array(pred_3d[:num_frames])[:, mp_idx]
    g_subset = np.array(gt_3d[:num_frames])[:, gt_idx]
    
    p_aligned = procrustes_alignment_batched(p_subset, g_subset)
    dist = np.linalg.norm(p_aligned - g_subset, axis=-1)
    return np.mean(dist)


# 6. Define MediaPipe 3D Processing Function

# 7. Validation Evaluation Pipeline

## 7.1 Generate Validation Tasks

In [6]:
def get_validation_tasks(dataset_base, subj_names, all_camera_names):
    tasks = []
    base_path = Path(dataset_base)
    for subj in subj_names:
        ref_dir = base_path / "train" / subj / "joints3d_25"
        if not ref_dir.exists(): continue
        for ref_file in ref_dir.glob("*.json"):
            action = ref_file.stem
            gt_3d = None
            for cam_id in np.random.choice(all_camera_names, 1, replace=False):
                v_path = base_path / "train" / subj / "videos" / str(cam_id) / f"{action}.mp4"
                if v_path.exists():
                    if gt_3d is None:
                        with open(ref_file, 'r') as f:
                            gt_data = json.load(f)
                            gt_3d = np.array(gt_data.get('joints3d_25', list(gt_data.values())[0])) if isinstance(gt_data, dict) else np.array(gt_data)
                    tasks.append({'subj': subj, 'action': action, 'cam_id': cam_id, 'video_path': str(v_path), 'gt_3d': gt_3d})
    return tasks

## 7.2 Validation Execution Pipeline

In [7]:
def evaluate_validation_set(tasks, predict_func, complexity, mp_indices, fit3d_indices):
    print(f"Starting parallel validation evaluation of {len(tasks)} videos...", flush=True)
    all_pa_mpjpe = []
    all_fps = []
    action_pa_mpjpe = {}
    
    def process_task(t):
        result = predict_func(t['video_path'], complexity)
        if result is None or result[0] is None: return None
        pred_3d_raw, average_fps = result[:2]
        
        gt_3d_mm   = t['gt_3d'] * 1000
        pred_3d_mm = pred_3d_raw * 1000.0
        try:
            pa = calculate_pa_mpjpe(pred_3d_mm, gt_3d_mm, mp_indices, fit3d_indices)
            return (t['subj'], t['action'], t['cam_id'], pa, average_fps)
        except Exception as e:
            return ('ERROR', t['subj'], t['action'], t.get('cam_id', 'TestCam'), e, 0.0)
        
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(process_task, t) for t in tasks]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks), desc="Evaluating Pose"):
            res = future.result()
            if isinstance(res, tuple) and res[0] == 'ERROR':
                print(f"\nERROR processing {res[1]} | {res[2]} (Cam {res[3]}): {res[4]}")
            else:
                subj, action, cam_id, pa, average_fps = res
                all_pa_mpjpe.append(pa)
                action_pa_mpjpe.setdefault(action, []).append(pa)
                if average_fps > 0:
                    all_fps.append(average_fps)
                
    if all_pa_mpjpe:
        print("\n\n--- PA-MPJPE per Exercise ---")
        for act in sorted(action_pa_mpjpe.keys()):
            print(f"{act:15s}: PA-MPJPE = {np.mean(action_pa_mpjpe[act]):6.2f} mm")
        print(f"\nOverall Mean PA-MPJPE: {np.mean(all_pa_mpjpe):.2f} mm")
        if all_fps:
            print(f"Overall Mean FPS: {np.mean(all_fps):.2f} FPS")
    return all_pa_mpjpe


# 8. Define MediaPipe 2D Processing Function

In [8]:
def predict_mediapipe_2d(video_path, complexity):
    pred_2d_list = []
    # Force OpenCV to use Hardware Acceleration for video decoding
    cap = cv2.VideoCapture(video_path, cv2.CAP_ANY, [cv2.CAP_PROP_HW_ACCELERATION, cv2.VIDEO_ACCELERATION_ANY])
    total_inference_time = 0.0
    total_frames = 0
    
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=complexity, static_image_mode=False) as pose:
        last_pose = np.zeros((33, 3))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
                
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            
            # 1. Start the high-precision timer
            start_time = time.perf_counter()

            # 2. Run ONLY the model inference
            results = pose.process(image)

            # 3. Stop the timer
            end_time = time.perf_counter()

            # 4. Calculate exactly how long this one frame took
            frame_time = end_time - start_time
            total_inference_time += frame_time
            total_frames += 1
            height, width, _ = frame.shape
            
            if results.pose_landmarks:
                # 2D relative landmarks scaled to image dimensions
                lms = np.array([[lm.x * width, lm.y * height, lm.z * width] for lm in results.pose_landmarks.landmark])
                pred_2d_list.append(lms)
                last_pose = lms
            else:
                pred_2d_list.append(last_pose)
    
    cap.release()
    average_fps = total_frames / total_inference_time if total_inference_time > 0 else 0.0
    if len(pred_2d_list) == 0:
        return None, average_fps, 0, 0
    return np.array(pred_2d_list), average_fps, width, height

# 9. Execute Experiments (PA-MPJPE)

In [9]:
val_tasks = get_validation_tasks(dataset_Folder, val_subj_names, all_camera_names)

In [10]:
# 10. PoseFormer 3D Conversion and Evaluation
import sys
sys.path.insert(0, r'd:\GP\Pose\PoseFormer')
from common.model_poseformer import PoseTransformer
import torch

# Official pre-trained model: 81-frame, CPN detected 2D pose as input
num_frame = 81
pose_model = PoseTransformer(
    num_frame=num_frame, num_joints=17, in_chans=2,
    embed_dim_ratio=32, depth=4, num_heads=8,
    mlp_ratio=2., qkv_bias=True, qk_scale=None, drop_path_rate=0
)

# Load official pre-trained weights (detected81f.bin - CPN detected 2D -> 3D)
checkpoint_path = r'd:\GP\Pose\PoseFormer\checkpoint\detected81f.bin'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
pose_model.load_state_dict(checkpoint['model_pos'], strict=False)
print(f"Loaded official PoseFormer weights from epoch {checkpoint['epoch']}")

if torch.cuda.is_available():
    pose_model = pose_model.cuda()
pose_model.eval()


def mediapipe_to_h36m(np_lms, img_w, img_h):
    """
    Convert MediaPipe 33-joint landmarks to H36M 17-joint 2D keypoints.
    Normalizes to [-1, 1] to match the official PoseFormer training distribution.
    np_lms: (N, 33, 3) with x,y in pixel coords
    """
    N = np_lms.shape[0]
    h36m = np.zeros((N, 17, 2))
    lms = np_lms[..., :2]  # x, y in pixels

    h36m[:, 1] = lms[:, 24]  # R_Hip
    h36m[:, 2] = lms[:, 26]  # R_Knee
    h36m[:, 3] = lms[:, 28]  # R_Ankle
    h36m[:, 4] = lms[:, 23]  # L_Hip
    h36m[:, 5] = lms[:, 25]  # L_Knee
    h36m[:, 6] = lms[:, 27]  # L_Ankle
    h36m[:, 0] = (lms[:, 23] + lms[:, 24]) / 2.0  # Pelvis

    h36m[:, 11] = lms[:, 11]  # L_Shoulder
    h36m[:, 12] = lms[:, 13]  # L_Elbow
    h36m[:, 13] = lms[:, 15]  # L_Wrist
    h36m[:, 14] = lms[:, 12]  # R_Shoulder
    h36m[:, 15] = lms[:, 14]  # R_Elbow
    h36m[:, 16] = lms[:, 16]  # R_Wrist

    h36m[:, 8] = (lms[:, 11] + lms[:, 12]) / 2.0  # Neck
    h36m[:, 7] = (h36m[:, 0] + h36m[:, 8]) / 2.0  # Spine
    h36m[:, 9] = lms[:, 0]   # Nose
    h36m[:, 10] = lms[:, 0]  # Head top approx

    # Normalize pixel coords to [-1, 1] (matches official normalize_screen_coordinates)
    h36m[..., 0] = h36m[..., 0] / img_w * 2 - 1
    h36m[..., 1] = h36m[..., 1] / img_h * 2 - 1

    return h36m


def predict_poseformer(video_path, complexity):
    # Step 1: Get 2D predictions from MediaPipe
    result = predict_mediapipe_2d(video_path, complexity)
    if result[0] is None:
        return None, result[1]
    pred_2d_raw, average_fps, img_w, img_h = result

    # Step 2: Map to H36M 17 joints, normalized to [-1, 1]
    h36m_2d = mediapipe_to_h36m(pred_2d_raw, img_w, img_h)  # (num_frames, 17, 2)

    # Step 3: Sliding window inference with edge padding
    pad = num_frame // 2
    padded = np.pad(h36m_2d, ((pad, pad), (0, 0), (0, 0)), mode='edge')

    num_frames = h36m_2d.shape[0]
    total_mp_time = num_frames / average_fps if average_fps > 0 else 0

    import time
    start_pf = time.perf_counter()
    
    # --- OPTIMIZED BATCHED INFERENCE ---
    # Convert overlapping sliding windows into a batch tensor
    windows = np.array([padded[i:i + num_frame] for i in range(num_frames)])
    windows_tensor = torch.tensor(windows).float()
    
    pred_3d_list = []
    batch_size = 256 # Process in batches to parallelize and save GPU memory
    
    with torch.no_grad():
        for i in range(0, num_frames, batch_size):
            batch = windows_tensor[i:i + batch_size]
            if torch.cuda.is_available():
                batch = batch.cuda()

            # Forward pass over the batch. Output: (B, 1, 17, 3)
            out_3d = pose_model(batch) 
            pred_3d_list.append(out_3d.squeeze(1).cpu().numpy()) # Squeeze temporal output dim

    # Concat batches along temporal axis
    pred_3d = np.concatenate(pred_3d_list, axis=0)  # (num_frames, 17, 3) in meters

    end_pf = time.perf_counter()
    total_pf_time = end_pf - start_pf
    
    combined_fps = num_frames / (total_mp_time + total_pf_time) if (total_mp_time + total_pf_time) > 0 else 0

    return pred_3d, combined_fps


poseformer_indices = [4, 5, 6, 1, 2, 3, 11, 12, 13, 14, 15, 16]
print("========== Experiment: PoseFormer (MediaPipe 2D -> 3D with Official Weights) ==========")
# _ = evaluate_validation_set(val_tasks, predict_poseformer, complexity=1, mp_indices=poseformer_indices, fit3d_indices=fit3d_indices)


d:\GP\Pose\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\GP\Pose\.venv\Lib\site-packages\timm\models\helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
d:\GP\Pose\.venv\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
d:\GP\Pose\.venv\Lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please impo

Loaded official PoseFormer weights from epoch 78
========== Experiment: PoseFormer (MediaPipe 2D -> 3D with Official Weights) ==========


# 11. PoseFormer Training on Fit3D


In [11]:
def fit3d_to_h36m(gt_3d_25):
    N = gt_3d_25.shape[0]
    h36m = np.zeros((N, 17, 3))
    # Lower body
    h36m[:, 0] = gt_3d_25[:, 0]
    h36m[:, 1] = gt_3d_25[:, 4]
    h36m[:, 2] = gt_3d_25[:, 5]
    h36m[:, 3] = gt_3d_25[:, 6]
    h36m[:, 4] = gt_3d_25[:, 1]
    h36m[:, 5] = gt_3d_25[:, 2]
    h36m[:, 6] = gt_3d_25[:, 3]
    # Arms
    h36m[:, 11] = gt_3d_25[:, 11]
    h36m[:, 12] = gt_3d_25[:, 12]
    h36m[:, 13] = gt_3d_25[:, 13]
    h36m[:, 14] = gt_3d_25[:, 14]
    h36m[:, 15] = gt_3d_25[:, 15]
    h36m[:, 16] = gt_3d_25[:, 16]
    # Derived
    h36m[:, 8] = (gt_3d_25[:, 11] + gt_3d_25[:, 14]) / 2.0
    h36m[:, 7] = (h36m[:, 0] + h36m[:, 8]) / 2.0
    h36m[:, 9] = gt_3d_25[:, 9]
    h36m[:, 10] = gt_3d_25[:, 10]
    return h36m


In [12]:
class Fit3DDatasetH5(Dataset):
    def __init__(self, h5_path, num_frame=81):
        self.h5_path = Path(h5_path)
        self.num_frame = num_frame
        self.samples = []
        
        if not self.h5_path.exists():
            print(f"Warning: No HDF5 file found at {self.h5_path}")
            return
            
        # Only load the indices into RAM, not the full dataset
        with h5py.File(self.h5_path, 'r') as f:
            for key in f.keys():
                n_frames = f[key]['lms_2d'].shape[0]
                stride = num_frame // 2
                for i in range(0, n_frames - num_frame + 1, stride):
                    self.samples.append((key, i)) # Store video key and start frame
                    
    def __len__(self): 
        return len(self.samples)
        
    def __getitem__(self, idx):
        key, start_idx = self.samples[idx]
        end_idx = start_idx + self.num_frame
        
        # Lazy loading: read only the specific 81-frame window from disk
        with h5py.File(self.h5_path, 'r') as f:
            lms_2d = f[key]['lms_2d'][start_idx:end_idx]
            gt_3d = f[key]['gt_3d'][start_idx:end_idx]
        
        gt_3d = gt_3d - gt_3d[:, :1, :] # Root relative
        return torch.from_numpy(lms_2d).float(), torch.from_numpy(gt_3d).float()

In [13]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def train_poseformer(model, train_loader, val_loader=None, epochs=100, device='cuda', patience=10):
    print(f"Training for {epochs} epochs on {device}...")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # 1. ReduceLROnPlateau
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    history_train = []
    history_val = []
    
    poseformer_indices = [4, 5, 6, 1, 2, 3, 11, 12, 13, 14, 15, 16]
    fit3d_indices = [1, 2, 3, 4, 5, 6, 11, 12, 13, 14, 15, 16]
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for inputs_2d, targets_3d in train_loader:
            inputs_2d = inputs_2d.to(device, non_blocking=True)
            targets_3d = targets_3d.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs_3d = model(inputs_2d)
            target_center = targets_3d[:, 40:41, :, :]
            loss = torch.mean(torch.norm(outputs_3d - target_center, dim=-1))
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            
        avg_train_loss = total_train_loss / len(train_loader)
        history_train.append(avg_train_loss)
        
        # Validation Loop (PA-MPJPE)
        avg_val_pa_mpjpe = 0.0
        if val_loader:
            model.eval()
            val_pa_mpjpes = []
            with torch.no_grad():
                for inputs_2d, targets_3d in val_loader:
                    inputs_2d, targets_3d = inputs_2d.to(device), targets_3d.to(device)
                    outputs_3d = model(inputs_2d)
                    target_center = targets_3d[:, 40:41, :, :]
                    
                    pred_np = outputs_3d.squeeze(1).cpu().numpy() * 1000.0  # mm
                    gt_np = target_center.squeeze(1).cpu().numpy() * 1000.0 # mm
                    
                    # Calculate PA-MPJPE precisely using our function
                    pa = calculate_pa_mpjpe(pred_np, gt_np, poseformer_indices, fit3d_indices)
                    if pa > 0: val_pa_mpjpes.append(pa)
            
            if len(val_pa_mpjpes) > 0:
                avg_val_pa_mpjpe = np.mean(val_pa_mpjpes)
                history_val.append(avg_val_pa_mpjpe)
                
                # Step the LR Scheduler
                scheduler.step(avg_val_pa_mpjpe)
                
                # 2. Early Stopping Logic
                if avg_val_pa_mpjpe < best_val_loss:
                    best_val_loss = avg_val_pa_mpjpe
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), 'best_poseformer.pth')
                else:
                    epochs_no_improve += 1
                    
                if epochs_no_improve >= patience:
                    print(f"\nEarly stopping triggered at Epoch {epoch+1}! Best PA-MPJPE: {best_val_loss:.2f} mm")
                    break

        # Output progress
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val PA-MPJPE: {avg_val_pa_mpjpe:.2f} mm")
        
        # 3. Dynamic Graphing
        if val_loader and epoch > 0:
            clear_output(wait=True)
            plt.figure(figsize=(10, 5))
            plt.plot(range(1, len(history_val) + 1), history_val, label='Validation PA-MPJPE (mm)', color='orange', marker='o')
            plt.title('PoseFormer Fine-Tuning: PA-MPJPE vs Epochs')
            plt.xlabel('Epochs')
            plt.ylabel('PA-MPJPE (millimeters)')
            plt.grid(True)
            plt.legend()
            plt.show()

    return history_train, history_val

In [14]:
import h5py
import numpy as np
import json
import concurrent.futures
from pathlib import Path
from tqdm import tqdm

# 1. Modified single video processor (Returns data instead of saving to disk)
def process_single_video_cache(args):
    subj, action, cam_id, ref_file, v_path = args

    # Run Mediapipe
    result = predict_mediapipe_2d(str(v_path), complexity=1)
    if result[0] is None: return None
    lms_2d_raw, _, w, h = result

    # Load Ground Truth
    with open(ref_file, "r") as f:
        gt_data = json.load(f)
        gt_3d_25 = np.array(gt_data.get("joints3d_25", list(gt_data.values())[0]))

    # Convert formats
    h36m_gt_3d = fit3d_to_h36m(gt_3d_25)
    lms_2d = mediapipe_to_h36m(lms_2d_raw, w, h)
    n_frames = min(lms_2d.shape[0], h36m_gt_3d.shape[0])

    # Create a unique key name for this sample
    sample_name = f"{subj}_{action}_{cam_id}"

    # Return the processed arrays back to the main thread
    return sample_name, lms_2d[:n_frames].astype(np.float32), h36m_gt_3d[:n_frames].astype(np.float32)


# 2. Modified Extractor (Appends directly to HDF5)
def extract_training_data_to_hdf5(dataset_base, subj_names, cam_names, hdf5_path):
    base_path = Path(dataset_base)
    
    # Check what is already inside the HDF5 file so we don't repeat work
    existing_keys = set()
    if Path(hdf5_path).exists():
        with h5py.File(hdf5_path, 'r') as hf:
            existing_keys = set(hf.keys())
            
    tasks = []
    for subj in subj_names:
        ref_dir = base_path / "train" / subj / "joints3d_25"
        if not ref_dir.exists(): continue
        
        for ref_file in ref_dir.glob("*.json"):
            action = ref_file.stem
            for cam_id in cam_names:
                sample_name = f"{subj}_{action}_{cam_id}"
                
                # SKIP if we already packed this from the .npz files!
                if sample_name in existing_keys: 
                    continue
                
                v_path = base_path / "train" / subj / "videos" / str(cam_id) / f"{action}.mp4"
                if not v_path.exists(): continue
                
                tasks.append((subj, action, cam_id, ref_file, v_path))
                
    print(f"Found {len(tasks)} NEW videos to process.")
    if not tasks: return
    
    # Open HDF5 file in 'append' ('a') mode
    with h5py.File(hdf5_path, 'a') as hf:
        max_cores = max(1, os.cpu_count() - 2) 
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_cores) as executor:
            futures = [executor.submit(process_single_video_cache, t) for t in tasks]
            
            for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks), desc="Extracting to HDF5"):
                try:
                    res = future.result()
                    if res is not None:
                        sample_name, lms_2d, gt_3d = res
                        
                        # The main thread safely writes to the HDF5 file
                        group = hf.create_group(sample_name)
                        group.create_dataset("lms_2d", data=lms_2d, compression="lzf")
                        group.create_dataset("gt_3d", data=gt_3d, compression="lzf")
                        
                        # Forces HDF5 to write the buffer to the hard drive immediately
                        hf.flush() 

                except Exception as e:
                    print(f"Error processing task: {e}")

In [ ]:
if __name__ == '__main__':
    train_h5_path = "d:/GP/Pose/Dataset/train_dataset.h5"
    val_h5_path = "d:/GP/Pose/Dataset/val_dataset.h5"

    # 1. Extract directly to HDF5 files (this will skip videos that are already inside the HDF5!)
    extract_training_data_to_hdf5(dataset_Folder, val_subj_names, all_camera_names, val_h5_path)
    extract_training_data_to_hdf5(dataset_Folder, all_train, all_camera_names, train_h5_path)

    # 2. Load the data using the new HDF5 Dataset class
    train_dataset = Fit3DDatasetH5(train_h5_path)
    val_dataset = Fit3DDatasetH5(val_h5_path)

    if len(train_dataset) > 0 and len(val_dataset) > 0:
        # 3. Optimize DataLoader with pin_memory and multiple workers
        train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, pin_memory=True, num_workers=4)
        val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, pin_memory=True, num_workers=4)
        
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        # 4. Start Training
        train_poseformer(pose_model, train_loader, val_loader=val_loader, epochs=50, device=device, patience=8)
    else:
        print(f"Warning: Datasets empty. Train={len(train_dataset)}, Val={len(val_dataset)}. Run extraction!")

Found 0 NEW videos to process.
Found 599 NEW videos to process.


Extracting to HDF5:   0%|          | 0/599 [00:00<?, ?it/s]